## Overview

In this notebook, we will utilize the AIEnrichment Service to execute the TA4H, transforming it into the standard AIEnrichment Output schema. This process includes several steps: configuration management, model setup, input preparation, and execution of the enrichment service.

#### Prerequisites
- **TA4H Deployment:** Ensure you have a deployed Text Analytics for Health API. Refer to the TA4H documentation [here](https://learn.microsoft.com/en-us/azure/ai-services/language-service/text-analytics-for-health/overview).

#### Input Preparation
- Define the Enrichment Definition to prepare the enrichment input data.
- Specify metadata and text file references for the enrichment process.

#### AI Enrichment Service Execution
- Execute the enrichment service using the Enrichment ID.

#### Configurations
Before running this notebook, ensure that all configurations in the msft_config_notebook are completed.

###### Configuration management and setup

To setup and manage configurations for the Healthcare data solutions, please execute the following cell:

In [ ]:
%run msft_config_notebook

In [ ]:
%run msft_config_notebook {"enable_spark_setup" : true, "enable_packages_mount" : false}

In [ ]:
from microsoft.fabric.hls.hds.ai_enrichments.use_cases import TA4HTransformer,TA4HModelProcessor
from microsoft.fabric.hls.hds.ai_enrichments.core.services.ai_enrichments_service import AIEnrichmentsService
from microsoft.fabric.hls.hds.ai_enrichments.core import EnrichmentView,Enrichment,EnrichmentViewExpression,EnrichmentDefinition,EnrichmentInputMapping,EnrichmentFileReference,EnrichmentViewDefinition
from microsoft.fabric.hls.hds.ai_enrichments.core.models.enrichment.output.fhir_resource_transform_config import FHIRResourceTransformConfig


##### Configurations for TA4H Enrichment

In [ ]:
# Inline Parameters
INLINES_PARAMS={ 
        'refresh-metadata':True # Set to TRUE on the initial run or whenever new tables are added to silver
}

In [ ]:
# This configuration specifies the properties to be included in the transformation for each FHIR resource type.
FHIR_RESOURCES_CONFIG=dict(
                Observation = FHIRResourceTransformConfig(
                    properties = [
                        "id", "resourceType","status", "extension", "category", "code", "subject", 
                        "effectiveDateTime", "valueQuantity"
                    ]
                ),
                Condition = FHIRResourceTransformConfig(
                    properties = [
                        "id","resourceType", "extension", "clinicalStatus", "verificationStatus", "category", 
                        "code", "subject", "onsetDateTime"
                    ]
                )
            )

##### Initialize Processor and Transformer

In [ ]:
ta4h_processor=TA4HModelProcessor()
ta4h_transformer=TA4HTransformer(fhir_resources_config=FHIR_RESOURCES_CONFIG)

##### Initialize AIEnrichment Service

In [ ]:
#AI Enrichment Service
ai_enrichments_service=AIEnrichmentsService(
        spark,
        workspace_name=workspace_name,
        solution_name=solution_name,
        admin_lakehouse_name=administration_database_name,
        inline_params=INLINES_PARAMS,
        enrichment_model_processor=ta4h_processor,
        enrichment_transformer=ta4h_transformer,
        one_lake_endpoint=one_lake_endpoint)



##### Create View Definition

In [ ]:
# Define the tables to be used in the enrichment view
ENRICHMENT_VIEW_TABLES = ["Patient", "DocumentReference", "DocumentReferenceContent"]

# Define the SQL query to create the enrichment view
SQL_QUERY = """
SELECT 
    c.id AS document_id,
    c.content_attachment_data, 
    get_json_object(d.subject_string, '$.id') AS silver_patient_id,
    p.idOrig as patient_id,
    d.msftSourceSystem as source_system
FROM 
    view3 c 
JOIN 
    view2 d ON d.id = c.id 
JOIN 
    view1 p ON p.id = get_json_object(d.subject_string, '$.id')
"""

PARENT_VIEW_IDS = ai_enrichments_service.metadata.get_enrichment_view_ids(ENRICHMENT_VIEW_TABLES)

In [ ]:
sql_expression=EnrichmentViewExpression(
    type="sql",
    query=SQL_QUERY
)

enrichment_view_definition=EnrichmentViewDefinition(
    parent_views_ids=PARENT_VIEW_IDS,
    expression=sql_expression
)

enrichment_view = EnrichmentView(      
    name='View for TA4H Execution',  
    description='Creating Enrichment view Definition of Clinical Notes',  
    definition=enrichment_view_definition
)  

enrichment_view_id=ai_enrichments_service.metadata.create_enrichment_view(enrichment_view)

##### Create Enrichment Definition

In [ ]:
# Define TA4H API settings 
TA4H_API_KEY_SECRET_NAME = ""  # Secret Key Name for TA4H service Endpoint API Key
TA4H_API_ENDPOINT = ""  # API endpoint for TA4H service
TA4H_API_VERSION = "2023-11-15-preview"  # API version for TA4H service
TA4H_MODEL_VERSION = "2023-12-01"  # Model version for TA4H service
TA4H_MODEL_NAME="TA4H Model" # Model Name for TA4H service
TA4H_FHIR_VERSION="4.0.1" # FHIR version for TA4H service to be used generating FHIR resources

# Model configuration for TA4H Enrichment
model_definition = {  
    "api_key_secret_name": TA4H_API_KEY_SECRET_NAME,  
    "api_endpoint": TA4H_API_ENDPOINT,  
    "api_version": TA4H_API_VERSION,  
    "version": TA4H_MODEL_VERSION,
    "fhir_version":TA4H_FHIR_VERSION,
    "name":TA4H_MODEL_NAME
}

In [ ]:
metadata_dict={
    "document_id": "document_id",
    "source_system": "source_system",
}
  
column_references=EnrichmentFileReference(  
                    id="document_id",  
                    content="content_attachment_data"  
                ) 
input_mapping=EnrichmentInputMapping(  
            patient_id="patient_id",
            metadata=metadata_dict,  
            text_resource_references=[column_references]  
        )  

# Create an instance of Enrichment  
enrichment = Enrichment(  
    name='Enrichment for TA4H Execution',  
    description='Enrichment definition for Clinical Notes with TA4H',  
    definition=EnrichmentDefinition(  
        model=model_definition,  
        view_id=f"{enrichment_view_id}",  
        input_mapping=input_mapping
    )  
)  

enrichment_id=ai_enrichments_service.metadata.create_enrichment(enrichment)

#### AIEnrichment Execution

In [ ]:
ai_enrichments_service.execution.execute(enrichment_id)